# 构建 Makemore - 练习

练习来自 [构建 makemore 视频](https://www.youtube.com/watch?v=PaCmpygFfXo)。<br>
视频描述中包含了练习题目，同时也列在下方。

1. 在 YouTube 上观看 [构建 makemore 视频](https://www.youtube.com/watch?v=PaCmpygFfXo)
2. 回来完成练习以提升水平 :)

In [31]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
%matplotlib inline

## 练习 1 - 三元语言模型

**目标：** 训练一个三元语言模型（Trigram Language Model），即输入两个字符来预测第三个字符。<br>
可以使用计数法或神经网络来实现。评估损失值；它是否比二元模型有所改进？

In [ ]:
# Set training device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# Load dataset -> List[str]
words = open('../names.txt', 'r').read().splitlines()
g = torch.Generator(device=device).manual_seed(2147483647)

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0 # Special token has position zero
itos = {i:s for s,i in stoi.items()}

# TODO: Modify this to accommodate for trigrams
for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    # Two char 'sliding-window'
    for ch1, ch2 in zip(chs, chs[1:]):
        print(ch1, ch2)

# -----
# TODO: Your code here
# Implement a trigram model
# -----

## 练习 2 - 划分数据集，以及在验证集和测试集上评估

**目标：** 将数据集随机划分为 $80\\%$ 的 `训练集`、$10\\%$ 的 `验证集` 和 $10\\%$ 的 `测试集`。<br>
仅在 `训练集` 上训练二元模型和三元模型。分别在 `验证集` 和 `测试集` 上评估它们。

你能观察到什么？

In [5]:
g = torch.Generator(device=device).manual_seed(2147483647)

### 使用二元模型作为基准

我们使用视频中构建的二元模型代码来建立基准线。

In [ ]:
# Create set of all *bigrams*
xs, ys = [], []

for w in words:
    chs = ['.'] + list(w) + ['.']
    # Two char 'sliding-window'
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])

xs, ys = torch.tensor(xs), torch.tensor(ys) # [196113], [196113]
num_x, num_y = xs.nelement(), ys.nelement()

# TODO: Shuffle/Permute the dataset, keeping pairs in sync
# TODO: Split the dataset into 80:10:10 for train:valid:test
xs_bi_train, xs_bi_valid, xs_bi_test = None, None, None
ys_bi_train, ys_bi_valid, ys_bi_test = None, None, None

In [ ]:
W = torch.randn((27,27), device=device, generator=g, requires_grad=True)

# Training cycles, using the entire dataset -> 200 Epochs
for k in range(200):    
    # Forward pass
    xenc = F.one_hot(xs_bi_train, num_classes=27).float().to(device) # one-hot encode the names
    logits = xenc @ W # logits, different word for log-counts
    counts = logits.exp() # 'fake counts', kinda like in  the N matrix of bigram
    probs = counts / counts.sum(1, keepdims=True) # Normal distribution probabilities (this is y_pred)
    loss = -probs[torch.arange(len(probs)), ys_bi_train].log().mean() + 0.01 * (W**2).mean()
    print(f'Loss @ iteration {k+1}: {loss}')
    # Backward pass
    W.grad = None # Make sure all gradients are reset
    loss.backward() # Torch kept track of what this variable is, kinda cool
    # Weight update
    W.data += -50 * W.grad

In [ ]:
# Validation Loss
with torch.no_grad():
    xenc = F.one_hot(xs_bi_valid, num_classes=27).float().to(device) # one-hot encode the names
    logits = xenc @ W # logits, different word for log-counts
    counts = logits.exp() # 'fake counts', kinda like in  the N matrix of bigram
    probs = counts / counts.sum(1, keepdims=True) # Normal distribution probabilities (this is y_pred)
    loss = -probs[torch.arange(len(probs)), ys_bi_valid].log().mean() + 0.01 * (W**2).mean()
print(f'Validation Loss: {loss}')

# Test Loss
with torch.no_grad():
    xenc = F.one_hot(xs_bi_test, num_classes=27).float().to(device) # one-hot encode the names
    logits = xenc @ W # logits, different word for log-counts
    counts = logits.exp() # 'fake counts', kinda like in  the N matrix of bigram
    probs = counts / counts.sum(1, keepdims=True) # Normal distribution probabilities (this is y_pred)
    loss = -probs[torch.arange(len(probs)), ys_bi_test].log().mean() + 0.01 * (W**2).mean()
print(f'Test Loss:\t {loss}')

### 比较二元模型和三元模型

In [1]:
# TODO: Create set of all *trigrams*
xs, ys = [], []

# TODO: Shuffle/Permute the dataset, keeping (x,y) pairs in sync
# TODO: Split the dataset into 80:10:10 for train:valid:test
xs_tri_train, xs_tri_valid, xs_tri_test = None, None, None
ys_tri_train, ys_tri_valid, ys_tri_test = None, None, None

In [ ]:
# TODO: Implement and train a trigram model

In [ ]:
# TODO: Evaluate the trigram model on the validation and test sets

## 练习 3 - 调整平滑强度

**目标：** 使用 *验证集* 来调整三元模型的平滑（或正则化）强度——即<br>
尝试多种可能性，看看哪种方案在验证集损失上表现最好。<br>
在调整强度时，训练集和验证集的损失有什么规律？<br>
取最佳设置的平滑参数，最终在测试集上评估一次。<br>
你能达到多好的损失值？

In [56]:
# TODO: Create set of all *trigrams*
xs, ys = [], []

# TODO: Shuffle/Permute the dataset, keeping (x,y) pairs in sync
# TODO: Split the dataset into 80:10:10 for train:valid:test
xs_tri_train, xs_tri_valid, xs_tri_test = None, None, None
ys_tri_train, ys_tri_valid, ys_tri_test = None, None, None

In [ ]:
# TODO: Build the hyperparameter search for regularization strength of the trigram model

## 练习 4 - 移除独热向量

**目标：** 我们发现独热向量（One-Hot Vector）仅仅是选择 $W$ 的某一行，因此显式生成这些向量显得很浪费。<br>
能否去掉 `F.one_hot` 的使用，改为直接索引 $W$ 的行？

In [ ]:
# TODO: Rewrite the training loop to delete F.one_hot

## 练习 5：使用 F.cross_entropy

**目标：** 查阅并使用 `F.cross_entropy` 来替代手动计算。你应该得到相同的结果。想想为什么我们更倾向于使用 `F.cross_entropy`？以下是 [F.cross_entropy 文档](https://pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)。

In [ ]:
# TODO: Rewrite the training loop from Ex. 4 to employ F.cross_entropy

## 练习 6：开放练习

**目标：** 自己想一个有趣/好玩的练习并完成它。

In [ ]:
# TODO: The stage is yours!